<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDCNNImageClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import numpy as np
import numpy.lib.stride_tricks as lst
import matplotlib.pyplot as plt
import math
import random
import re
import cv2
import csv
import pandas as pd
from google.colab.patches import cv2_imshow
%matplotlib inline

In [34]:
# Hyperparameters
START_SIZE = 256
BATCH_SIZE = 32
WIN_SIZE_L1 = (5, 5)
NUM_FEATURES_L1 = 16
MAX_POOLING_SHRINK_L1 = 4
WIN_SIZE_L2 = (5, 5)
NUM_FEATURES_L2 = 64
MAX_POOLING_SHRINK_L2 = 4
WIN_SIZE_L3 = (5, 5)
NUM_FEATURES_L3 = 128
MAX_POOLING_SHRINK_L3 = 2
LEAKY_RELU_CONSTANT = 0.01
EPOCHS = 44
NUM_FEATURES_L4 = 64
LR = 0.04

In [35]:
# Grabbing the Images & Labels
df = pd.read_csv('IOImageData.csv')
img_names = df['Image Name'].to_list()
labels = df['Label'].to_list()

# Shuffling them
# combined = list(zip(img_names, labels))
# random.shuffle(combined)
# img_names, labels = list(zip(*combined))
# img_names, labels = list(img_names), list(labels)

In [36]:
# UNCOMMENT DEBUGGING ARTIFICAT ABOVE
# Preprocessing Image
images = []
for img_name in img_names[:10]: # DELETE DEBUGGING ARTIFICAT LATER!!!
    img_path = f"{img_name}.png"
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        resized_img = cv2.resize(img, (START_SIZE, START_SIZE))
        images.append(resized_img)

X = np.array(images).astype(np.float32)
X /= np.max(X, keepdims=True)
Y = np.array(labels[:10], dtype=np.float32)
Y = Y.reshape(-1, 1)
# Debugging the loss plateau
BATCH_SIZE = 8 # DELETE DEBUGGING ARTIFICAT LATER!!!

In [37]:
# Creating Training Testing Split
n = int(0.8 * X.shape[0])
Xtr, Xte = X[:n], X[n:]
Ytr, Yte = Y[:n], Y[n:]

In [38]:
# PARAPARAMETER!!!
W1 = np.random.randn(*WIN_SIZE_L1, NUM_FEATURES_L1)
W1 *= np.sqrt(2/(W1.reshape(-1, W1.shape[-1]).shape[0]))
g1 = np.ones(NUM_FEATURES_L1)
b1 = np.zeros_like(g1)
W2 = np.random.randn(*WIN_SIZE_L2, NUM_FEATURES_L1, NUM_FEATURES_L2)
W2 *= np.sqrt(2/(W2.reshape(-1, W2.shape[-1]).shape[0]))
g2 = np.ones(NUM_FEATURES_L2)
b2 = np.zeros_like(g2)
W3 = np.random.randn(*WIN_SIZE_L3, NUM_FEATURES_L2, NUM_FEATURES_L3)
W3 *= np.sqrt(2/(W3.reshape(-1, W3.shape[-1]).shape[0]))
g3 = np.ones(NUM_FEATURES_L3)
b3 = np.zeros_like(g3)
W4 = np.random.randn(NUM_FEATURES_L3 * (START_SIZE // MAX_POOLING_SHRINK_L1 // MAX_POOLING_SHRINK_L2 // MAX_POOLING_SHRINK_L3) ** 2, NUM_FEATURES_L4)
W4 *= np.sqrt(2/(W4.reshape(-1, W4.shape[-1]).shape[0]))
g4 = np.ones(NUM_FEATURES_L4)
b4 = np.zeros_like(g4)
W5 = np.random.randn(NUM_FEATURES_L4, 1)
W5 *= np.sqrt(2/(W5.reshape(-1, W5.shape[-1]).shape[0]))

parameters = [W1, g1, b1, W2, g2, b2, W3, g3, b3, W4, g4, b4, W5]

In [39]:
for epoch in range(EPOCHS):
  for i in range(0, Xtr.shape[0] // BATCH_SIZE):
    ### Mini Batching
    ix = np.random.randint(0, Xtr.shape[0], (BATCH_SIZE,))
    xbatch = Xtr[i*BATCH_SIZE:i*BATCH_SIZE+BATCH_SIZE]
    ybatch = Ytr[i*BATCH_SIZE:i*BATCH_SIZE+BATCH_SIZE]

    ### Forward Pass
    ## Layer 1
    # Padding
    pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
    img_padded1 = np.pad(xbatch, ((0,0), pad1, pad1), "constant", constant_values=0)

    # Conv
    win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
    win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])

    # Linear
    preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
    preln1 = preln1.reshape(-1, Xtr.shape[1], Xtr.shape[1], NUM_FEATURES_L1)

    # Layer Norm
    lnmean1 = np.mean(preln1, axis = -1, keepdims = True)
    lnvar1 = np.var(preln1, axis = -1, keepdims = True)
    lnraw1 = (preln1 - lnmean1)/np.sqrt(lnvar1 + 1e-5)
    ln1 = g1 * lnraw1 + b1

    # ReLu
    rl1 = np.maximum(LEAKY_RELU_CONSTANT * ln1, ln1)

    # Max Pooling
    rl1t = rl1.transpose(0, 3, 1, 2)
    img_pieced1 = rl1t.reshape(-1, NUM_FEATURES_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
    pooled_img1 = np.max(img_pieced1, axis=(-1, -3))

    ## Layer 2
    # Padding
    pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
    img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)

    # Conv
    win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
    win_l2c = np.ascontiguousarray(win_l2).transpose(0, 2, 3, 4, 5, 1)
    win_l2r = win_l2c.reshape(-1, win_l2c.shape[-2] * win_l2c.shape[-1] * win_l2c.shape[-3])

    # Linear
    preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
    preln2 = preln2.reshape(-1, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)

    # Layer Norm
    lnmean2 = np.mean(preln2, axis = -1, keepdims = True)
    lnvar2 = np.var(preln2, axis = -1, keepdims = True)
    lnraw2 = (preln2 - lnmean2)/np.sqrt(lnvar2 + 1e-5)
    ln2 = g2 * lnraw2 + b2

    # ReLu
    rl2 = np.maximum(LEAKY_RELU_CONSTANT * ln2, ln2)

    # Max Pooling
    rl2t = rl2.transpose(0, 2, 3, 1)
    img_pieced2 = rl2t.reshape(-1, NUM_FEATURES_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2)
    pooled_img2 = np.max(img_pieced2, axis=(-1, -3))

    ## Layer 3
    # Padding
    pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
    img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)

    # Conv
    win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
    win_l3c = np.ascontiguousarray(win_l3).transpose(0, 2, 3, 4, 5, 1)
    win_l3r = win_l3c.reshape(-1, win_l3c.shape[-2] * win_l3c.shape[-1] * win_l3c.shape[-3])

    # Linear
    preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
    preln3 = preln3.reshape(-1, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)

    # Layer Norm
    lnmean3 = np.mean(preln3, axis = -1, keepdims = True)
    lnvar3 = np.var(preln3, axis = -1, keepdims = True)
    lnraw3 = (preln3 - lnmean3)/np.sqrt(lnvar3 + 1e-5)
    ln3 = g3 * lnraw3 + b3

    # ReLu
    rl3 = np.maximum(LEAKY_RELU_CONSTANT * ln3, ln3)

    # Max Pooling
    rl3t = rl3.transpose(0, 3, 1, 2)
    img_pieced3 = rl3t.reshape(-1, NUM_FEATURES_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3)
    pooled_img3 = np.max(img_pieced3, axis=(-1, -3))

    ## Layer 4
    img_flattened = pooled_img3.reshape(img_pieced3.shape[0], -1)
    h4 = img_flattened @ W4
    # mlp_data = h4.copy() I am removing it. It'll be too hard to train the word dataset unless I give Gemini d images which is slow

    # Layer Norm... sighs
    lnmean4 = h4.mean(-1, keepdims=True)
    lnvar4 = h4.var(-1, keepdims=True)
    lnraw4 = (h4 - lnmean4)/np.sqrt(lnvar4 + 1e-5)
    ln4 = g4 * lnraw4 + b4

    # ReLu
    rl4 = np.maximum(LEAKY_RELU_CONSTANT * ln4, ln4)

    ## Layer 5
    raw_pred = rl4 @ W5

    ## Sigmoid + Loss
    safe_raw_pred = np.clip(raw_pred, -250, 250)
    probs = 1/(1 + (np.e ** -(safe_raw_pred)))
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    loss = -(ybatch * np.log(probs + 1e-5) + (1 - ybatch) * np.log(1 - probs + 1e-5))

    ### Backward Pass

    ## Sigmoid + Loss
    draw_pred = (probs - ybatch) / xbatch.shape[0] # B, 1

    ## Layer 5
    dW5 = rl4.T @ draw_pred
    drl4 = draw_pred @ W5.T # B, 8

    ## Layer 4
    dln4 = np.where((rl4 >= 0), drl4, drl4 * LEAKY_RELU_CONSTANT) # B, 8
    dlnraw4 = g4 * dln4
    dg4 = (dln4 * lnraw4).sum(axis=0)
    db4 = dln4.sum(axis = 0)
    dh4 = (dlnraw4 - dlnraw4.mean(axis=-1, keepdims=True) - lnraw4 * (dlnraw4 * lnraw4).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar4 + 1e-5))
    dimg_flattened = dh4 @ W4.T
    dW4 = img_flattened.T @ dh4
    dpooled_img3 = dimg_flattened.reshape(pooled_img3.shape)

    ## Layer 3
    # Max Pooling
    pooled_img3_expanded = np.max(img_pieced3, axis=(-1, -3), keepdims=True)
    # cords = np.where(np.isin(img_pieced3, pooled_img3))
    # dimg_pieced3 = np.zeros_like(img_pieced3) ## FIX DIS DERIVATIVE!!!
    mask3 = (img_pieced3 == pooled_img3_expanded) / (img_pieced3 == pooled_img3_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced3 = mask3 * np.expand_dims(dpooled_img3, axis=(-1, -3))

    # if np.round(mask3.mean(), decimals=2) != 0.25:
      # print(f"WARNING 1: The mean of img_pieced3 == pooled_img3_expanded is NOT 0.25. It is {mask3.mean()}")

    # print(drl3t.shape)
    drl3t = dimg_pieced3.reshape(rl3t.shape)
    drl3 = drl3t.transpose(0, 2, 3, 1)

    # ReLu
    dln3 = np.where((ln3 >= 0), drl3, drl3 * 0.01)
    dg3 = np.sum(lnraw3 * dln3, axis = (0, 1, 2))
    dlnraw3 = dln3 * g3
    db3 = np.sum(dln3, axis = (0, 1, 2))

    # LayerNorm
    dpreln3 = (dlnraw3 - dlnraw3.mean(axis=-1, keepdims=True) - lnraw3 * (dlnraw3 * lnraw3).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar3 + 1e-5))

    # Linear
    dwin_l3r = dpreln3.reshape(-1, NUM_FEATURES_L3) @ W3.reshape(-1, NUM_FEATURES_L3).T
    dW3 = (win_l3r.T @ dpreln3.reshape(-1, NUM_FEATURES_L3)).reshape(W3.shape)

    # Conv
    dwin_l3c = dwin_l3r.reshape(win_l3c.shape)
    dwin_l3 = dwin_l3c.transpose(0, 5, 1, 2, 3, 4)
    dimg_padded3 = np.zeros_like(img_padded3)
    row,col = WIN_SIZE_L3
    for r in range(row):
      for c in range(col):
        dimg_padded3[:, :, r:win_l3.shape[2]+r, c:win_l3.shape[3]+c] += dwin_l3[:, :, :, :, r, c]


    # Padding
    padw3 = int((WIN_SIZE_L3[0] - 1)//2)
    padh3 = int((WIN_SIZE_L3[1] - 1)//2)
    dpooled_img2 = dimg_padded3[:, :, padw3:-padw3, padh3:-padh3]


    ## Layer 2
    # Max Pooling
    pooled_img2_expanded = np.max(img_pieced2, axis=(-1, -3), keepdims=True)
    mask2 = (img_pieced2 == pooled_img2_expanded) / (img_pieced2 == pooled_img2_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced2 = mask2 * np.expand_dims(dpooled_img2, axis=(-1, -3))

    # if np.round(mask2.mean(), decimals=2) != 1/16:
    #   print(f"WARNING 2: The mean of img_pieced2 == pooled_img2_expanded is NOT {1/16}. It is {mask2.mean()}")

    drl2t = dimg_pieced2.reshape(rl2t.shape)
    drl2 = drl2t.transpose(0, 3, 1, 2)

    # ReLu
    dln2 = np.where((ln2 >= 0), drl2, drl2 * 0.01)
    dg2 = np.sum(lnraw2 * dln2, axis = (0, 1, 2))
    dlnraw2 = dln2 * g2
    db2 = np.sum(dln2, axis = (0, 1, 2))

    # LayerNorm
    dpreln2 = (dlnraw2 - dlnraw2.mean(axis=-1, keepdims=True) - lnraw2 * (dlnraw2 * lnraw2).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar2 + 1e-5))

    # Linear
    dwin_l2r = dpreln2.reshape(-1, NUM_FEATURES_L2) @ W2.reshape(-1, NUM_FEATURES_L2).T
    dW2 = (win_l2r.T @ dpreln2.reshape(-1, NUM_FEATURES_L2)).reshape(W2.shape)

    # Conv
    dwin_l2c = dwin_l2r.reshape(win_l2c.shape)
    dwin_l2 = dwin_l2c.transpose(0, 5, 1, 2, 3, 4)
    dimg_padded2 = np.zeros_like(img_padded2)

    row,col = WIN_SIZE_L2
    for r in range(row):
      for c in range(col):
        dimg_padded2[:, :, r:win_l2.shape[2]+r, c:win_l2.shape[3]+c] += dwin_l2[:, :, :, :, r, c]


    # Padding
    padw2 = int((WIN_SIZE_L2[0] - 1)/2)
    padh2 = int((WIN_SIZE_L2[1] - 1)/2)
    dpooled_img1 = dimg_padded2[:, :, padw2:-padw2, padh2:-padh2]


    ## Layer 1
    # Max Pooling
    pooled_img1_expanded = np.max(img_pieced1, axis=(-1, -3), keepdims=True)
    mask1 = (img_pieced1 == pooled_img1_expanded) / (img_pieced1 == pooled_img1_expanded).sum(axis=(-1,-3), keepdims=True)
    dimg_pieced1 = mask1 * np.expand_dims(dpooled_img1, axis=(-1, -3))

    # if np.round(mask1.mean(), decimals=2) != 1/16:
    #   print(f"WARNING 3: The mean of img_pieced1 == pooled_img1_expanded is NOT {1/16}. It is {mask1.mean()}")

    drl1t = dimg_pieced1.reshape(rl1t.shape)
    drl1 = drl1t.transpose(0, 2, 3, 1)

    # ReLu
    dln1 = np.where((ln1 >= 0), drl1, drl1 * 0.01)
    dg1 = np.sum(lnraw1 * dln1, axis = (0, 1, 2))
    dlnraw1 = dln1 * g1
    db1 = np.sum(dln1, axis = (0, 1, 2))

    # LayerNorm
    dpreln1 = (dlnraw1 - dlnraw1.mean(axis=-1, keepdims=True) - lnraw1 * (dlnraw1 * lnraw1).mean(axis=-1, keepdims=True))/(np.sqrt(lnvar1 + 1e-5))

    # Linear
    dwin_l1r = dpreln1.reshape(-1, NUM_FEATURES_L1) @ W1.reshape(-1, NUM_FEATURES_L1).T
    dW1 = (win_l1r.T @ dpreln1.reshape(-1, NUM_FEATURES_L1)).reshape(W1.shape)

    # Conv
    dwin_l1 = dwin_l1r.reshape(win_l1.shape)
    dimg_padded1 = np.zeros_like(img_padded1)

    row,col = WIN_SIZE_L1
    for r in range(row):
      for c in range(col):
        dimg_padded1[:, r:win_l1.shape[1]+r, c:win_l1.shape[2]+c] += dwin_l1[:, :, :, r, c]

    # Padding
    padw1 = int((WIN_SIZE_L1[0] - 1)/2)
    padh1 = int((WIN_SIZE_L1[1] - 1)/2)
    dpooled_img1 = dimg_padded1[:, padw1:-padw1, padh1:-padh1]

    # Gradient Clipping to Stop Gradient Explosions
    def clip_grad(g, max_norm=1.0):
      norm = np.linalg.norm(g)
      if norm > max_norm:
          g = g * (max_norm / norm)
      return g

    gradients = [clip_grad(g) for g in [dW1, dg1, db1, dW2, dg2, db2, dW3, dg3, db3, dW4, dg4, db4, dW5]]

    ### Update PARAPARAMETERS!!!
    lr = LR if epoch < int(0.75 * EPOCHS) else LR/2
    for parameter, gradient in reversed(list(zip(parameters, gradients))):
      update = lr * gradient
      parameter -= update

      # if i == 0:
        # print(f'0th Update to Parameter Ratio of Epoch {epoch}: {np.linalg.norm(update) / np.linalg.norm(parameter)}')
    # if i == 0:
      # print("xbatch batch var:", xbatch.var(axis=0).mean())
      # print("rl1 batch var:", rl1.var(axis=0).mean())
      # print("rl2 batch var:", rl2.var(axis=0).mean())
      # print("rl3 batch var:", rl3.var(axis=0).mean())


    ### Tracking Stats
    if epoch == 0 and i == 0:
      print(f"Initial Loss: {loss.mean()}")

  print(f"Epoch {epoch + 1}: Loss {loss.mean()}")
  # print(f'{np.mean(rl1 <= 0)=}, {np.mean(rl2 <= 0)=}, {np.mean(rl3 <= 0)=}, {np.mean(rl4 <= 0)=}') # Dead Neuron Ratio
  # print(f'{preln1.var()=}, {preln2.var()=}, {preln3.var()=}')
  # print(f"{np.abs(raw_pred).mean()=}, {np.abs(rl4).mean()=}, {np.abs(h4).mean()=}, {np.abs(ln4).mean()=}")
  # print(f"{np.abs(dW1).mean()=}, {np.abs(dW2).mean()=}, {np.abs(dW3).mean()=}, {np.abs(dW4).mean()=}, {np.abs(dW5).mean()=}")


Initial Loss: 0.5998627278580685
Epoch 1: Loss 0.5998627278580685
Epoch 2: Loss 0.8843180654580876
Epoch 3: Loss 0.5707323940211249
Epoch 4: Loss 0.7935086273710621
Epoch 5: Loss 0.5769736429178004
Epoch 6: Loss 0.670310227627642
Epoch 7: Loss 0.574216509863908
Epoch 8: Loss 0.6149032713127771
Epoch 9: Loss 0.5705102820488406
Epoch 10: Loss 0.5781775690605697
Epoch 11: Loss 0.5686792896396596
Epoch 12: Loss 0.5515923936279137
Epoch 13: Loss 0.560180903233584
Epoch 14: Loss 0.5341499007540278
Epoch 15: Loss 0.5504711433287814
Epoch 16: Loss 0.5221608142734269
Epoch 17: Loss 0.5407705152601672
Epoch 18: Loss 0.5080063106417665
Epoch 19: Loss 0.5283907462100695
Epoch 20: Loss 0.4949023677777219
Epoch 21: Loss 0.5138819176700733
Epoch 22: Loss 0.4796504183882011
Epoch 23: Loss 0.4965006276384555
Epoch 24: Loss 0.46805493426203865
Epoch 25: Loss 0.4901756189049299
Epoch 26: Loss 0.44703855719390356
Epoch 27: Loss 0.47614579033018256
Epoch 28: Loss 0.4264882185500833
Epoch 29: Loss 0.4589486

In [40]:
g4.shape, dg4.shape, dln4.shape

((64,), (64,), (8, 64))

In [41]:
print(probs.flatten())
print(ybatch.flatten())
print(rl4.std(axis=0))   # variance ACROSS the 8 examples, per feature

[0.93635325 0.95635767 0.96483548 0.07533831 0.93055073 0.04561565
 0.93022586 0.89677134]
[1. 1. 1. 0. 1. 0. 1. 1.]
[7.62578649e-01 4.12445368e-03 4.95039605e-02 4.34577400e-01
 4.67189551e-01 1.24927853e-03 4.53559524e-03 4.61472522e-03
 5.33462889e-01 3.83991710e-01 1.58983073e+00 1.35042533e+00
 7.15595675e-01 3.28475690e-02 3.80972761e-01 9.69766118e-01
 5.57530504e-02 5.32431161e-02 2.01744752e-03 5.03824534e-02
 2.53483437e-03 2.28721849e-02 3.65856754e-03 6.64549399e-01
 3.66547131e-03 3.53401074e-01 4.04534496e-02 1.38994933e-01
 1.86057253e-03 5.16812867e-01 3.21953831e-01 2.75368199e-01
 1.63351084e-01 3.69497579e-01 2.05213360e-03 2.74369652e-01
 3.75445705e-01 6.15052941e-02 1.79423587e-01 6.65428565e-01
 2.90971067e-01 3.47399831e-03 3.73271347e-03 7.51705472e-01
 3.32221480e-03 3.45311782e-02 7.92560507e-01 3.77346466e-01
 2.93405560e-01 3.40822426e-01 4.94216500e-02 3.61374261e-03
 4.99689066e-01 3.82507760e-03 4.52827398e-01 2.55532217e-01
 5.33924186e-01 1.97650163e-0

In [42]:
print(((probs > 0.5) + 0.0).flatten())
print(ybatch.flatten())

[1. 1. 1. 0. 1. 0. 1. 1.]
[1. 1. 1. 0. 1. 0. 1. 1.]


In [43]:
]

SyntaxError: unmatched ']' (1890416573.py, line 1)

In [45]:
len('                      '), len('                                                       ')

(22, 55)

In [48]:
def eval_loss():
  losses = []
  for i in range(0, Xte.shape[0] // BATCH_SIZE):
    ### Mini Batching
    xbatch = Xte[i*BATCH_SIZE:i*BATCH_SIZE+BATCH_SIZE]
    ybatch = Yte[i*BATCH_SIZE:i*BATCH_SIZE+BATCH_SIZE]

    ### Forward Pass
    ## Layer 1
    # Padding
    pad1 = tuple((size-1)//2 for size in WIN_SIZE_L1) # Same Padding Calcs
    img_padded1 = np.pad(xbatch, ((0,0), pad1, pad1), "constant", constant_values=0)

    # Conv
    win_l1 = lst.sliding_window_view(img_padded1, window_shape = WIN_SIZE_L1, axis = (1,2))
    win_l1r = (np.ascontiguousarray(win_l1)).reshape(-1, win_l1.shape[-2] * win_l1.shape[-1])

    # Linear
    preln1 = win_l1r @ W1.reshape(-1, W1.shape[-1])
    preln1 = preln1.reshape(-1, Xtr.shape[1], Xtr.shape[1], NUM_FEATURES_L1)

    # Layer Norm
    lnmean1 = np.mean(preln1, axis = -1, keepdims = True)
    lnvar1 = np.var(preln1, axis = -1, keepdims = True)
    lnraw1 = (preln1 - lnmean1)/np.sqrt(lnvar1 + 1e-5)
    ln1 = g1 * lnraw1 + b1

    # ReLu
    rl1 = np.maximum(LEAKY_RELU_CONSTANT * ln1, ln1)

    # Max Pooling
    rl1t = rl1.transpose(0, 3, 1, 2)
    img_pieced1 = rl1t.reshape(-1, NUM_FEATURES_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1, rl1t.shape[-1] // MAX_POOLING_SHRINK_L1, MAX_POOLING_SHRINK_L1)
    pooled_img1 = np.max(img_pieced1, axis=(-1, -3))

    ## Layer 2
    # Padding
    pad2 = tuple((size-1)//2 for size in WIN_SIZE_L2)
    img_padded2 = np.pad(pooled_img1, ((0,0), (0,0), pad2, pad2), "constant", constant_values=0)

    # Conv
    win_l2 = lst.sliding_window_view(img_padded2, window_shape = WIN_SIZE_L2, axis = (2,3))
    win_l2c = np.ascontiguousarray(win_l2).transpose(0, 2, 3, 4, 5, 1)
    win_l2r = win_l2c.reshape(-1, win_l2c.shape[-2] * win_l2c.shape[-1] * win_l2c.shape[-3])

    # Linear
    preln2 = win_l2r @ W2.reshape(-1, W2.shape[-1])
    preln2 = preln2.reshape(-1, pooled_img1.shape[2], pooled_img1.shape[2], NUM_FEATURES_L2)

    # Layer Norm
    lnmean2 = np.mean(preln2, axis = -1, keepdims = True)
    lnvar2 = np.var(preln2, axis = -1, keepdims = True)
    lnraw2 = (preln2 - lnmean2)/np.sqrt(lnvar2 + 1e-5)
    ln2 = g2 * lnraw2 + b2

    # ReLu
    rl2 = np.maximum(LEAKY_RELU_CONSTANT * ln2, ln2)

    # Max Pooling
    rl2t = rl2.transpose(0, 2, 3, 1)
    img_pieced2 = rl2t.reshape(-1, NUM_FEATURES_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2, rl2t.shape[-1] // MAX_POOLING_SHRINK_L2, MAX_POOLING_SHRINK_L2)
    pooled_img2 = np.max(img_pieced2, axis=(-1, -3))

    ## Layer 3
    # Padding
    pad3 = tuple((size-1)//2 for size in WIN_SIZE_L3)
    img_padded3 = np.pad(pooled_img2, ((0,0), (0,0), pad3, pad3), "constant", constant_values=0)

    # Conv
    win_l3 = lst.sliding_window_view(img_padded3, window_shape = WIN_SIZE_L3, axis = (2,3))
    win_l3c = np.ascontiguousarray(win_l3).transpose(0, 2, 3, 4, 5, 1)
    win_l3r = win_l3c.reshape(-1, win_l3c.shape[-2] * win_l3c.shape[-1] * win_l3c.shape[-3])

    # Linear
    preln3 = win_l3r @ W3.reshape(-1, W3.shape[-1])
    preln3 = preln3.reshape(-1, pooled_img2.shape[2], pooled_img2.shape[2], NUM_FEATURES_L3)

    # Layer Norm
    lnmean3 = np.mean(preln3, axis = -1, keepdims = True)
    lnvar3 = np.var(preln3, axis = -1, keepdims = True)
    lnraw3 = (preln3 - lnmean3)/np.sqrt(lnvar3 + 1e-5)
    ln3 = g3 * lnraw3 + b3

    # ReLu
    rl3 = np.maximum(LEAKY_RELU_CONSTANT * ln3, ln3)

    # Max Pooling
    rl3t = rl3.transpose(0, 3, 1, 2)
    img_pieced3 = rl3t.reshape(-1, NUM_FEATURES_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3, rl3t.shape[-1] // MAX_POOLING_SHRINK_L3, MAX_POOLING_SHRINK_L3)
    pooled_img3 = np.max(img_pieced3, axis=(-1, -3))

    ## Layer 4
    img_flattened = pooled_img3.reshape(img_pieced3.shape[0], -1)
    h4 = img_flattened @ W4
    # mlp_data = h4.copy() I am removing it. It'll be too hard to train the word dataset unless I give Gemini d images which is slow

    # Layer Norm... sighs
    lnmean4 = h4.mean(-1, keepdims=True)
    lnvar4 = h4.var(-1, keepdims=True)
    lnraw4 = (h4 - lnmean4)/np.sqrt(lnvar4 + 1e-5)
    ln4 = g4 * lnraw4 + b4

    # ReLu
    rl4 = np.maximum(LEAKY_RELU_CONSTANT * ln4, ln4)

    ## Layer 5
    raw_pred = rl4 @ W5

    ## Sigmoid + Loss
    safe_raw_pred = np.clip(raw_pred, -250, 250)
    probs = 1/(1 + (np.e ** -(safe_raw_pred)))
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    loss = -(ybatch * np.log(probs + 1e-5) + (1 - ybatch) * np.log(1 - probs + 1e-5))
    losses.append(loss)

  losses = np.array(losses)
  print(f'Testing Eval Loss: {losses.mean()}')

In [49]:
eval_loss()

Testing Eval Loss: 0.0710205607874228


In [ ]:
save_parameters = input('Save Parameters (y/n): ')
paranames = ["W1", "g1", "b1", "W2", "g2", "b2", "W3", "g3", "b3", "W4", "W5"]
if save_parameters == 'y':
  np.savez('NeuroLearnImageCNNWeights.npz', *parameters, *paranames)